In [1]:
from sentence_transformers import SentenceTransformer
import sentence_transformers
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
embs = model.encode(["I love cricket", "Cricket is my favourite sport"])
print(embs.shape)   # (2, 384)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(2, 384)


In [2]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [8]:
import torch 
#torch.allclose(torch.tensor(cosine_sim(embs[0],embs[1])),torch.tensor(sentence_transformers.util.cos_sim(embs[0],embs[1])))
np.isclose(cosine_sim(embs[0],embs[1]),sentence_transformers.util.cos_sim(embs[0],embs[1]).item())

np.True_

In [4]:
docs = ["Luffy has the best mindset",
        "One Piece teaches you so much",
        "As long as I am alive I will have infinite chances",
        "I don't care if I can't do it, I want to so I am",
        "Machine Learning is interesting",
        "ML has a lot of math",
        "CNNs were easier than transformers",
        "Loss reduction is fun to watch",
        "Football is the best sport",
        "Attacking midfielder is my favourite position",
        "Making the run is important for Strikers",
        "A through ball goal is fun",
        "Dogs are cool",
        "I like having a pet but I won't have one",
        "Golden Retriever's are nice",
        "Dogs are loyal",
        "I like shoes",
        "Onitsuka Tiger has good shoes",
        "Naked Wolf have good shoe designs",
        "Sneakers are neat"]                       
doc_embs = model.encode(docs) # encode on the above sentences

def search(query, k=3): 
    q = model.encode([query])[0] # get the encoding for our query
    scores = [cosine_sim(q, d) for d in doc_embs] # get comparision scores against our sentences
    for i in np.argsort(scores)[::-1][:k]:
        print(f"{scores[i]:.3f}  {docs[i]}")


In [5]:
# my tests
search("Mugiwara has the nicest character")
search("Shoes are useless")
search("I don't like dogs")

0.340  Attacking midfielder is my favourite position
0.301  Luffy has the best mindset
0.285  Onitsuka Tiger has good shoes
0.599  Sneakers are neat
0.568  I like shoes
0.497  Onitsuka Tiger has good shoes
0.576  Dogs are cool
0.561  I like having a pet but I won't have one
0.501  Dogs are loyal


In [6]:
doc_n = doc_embs / np.linalg.norm(doc_embs, axis=1, keepdims=True) #normalize for cosine later

def search_fast(query, k=3):
    q = model.encode([query])[0]
    q = q / np.linalg.norm(q)
    scores = doc_n @ q # matrix mul gives dot products in this format
    for i in np.argsort(scores)[::-1][:k]:  # print top 3 scores
        print(f"{scores[i]:.3f}  {docs[i]}")

In [7]:
search("I don't like dogs")
print()
search_fast("I don't like dogs")   # must match the lines above

print()
search_fast("I enjoy scoring after a clever pass")
print()
search_fast("friendly canine breed")

0.576  Dogs are cool
0.561  I like having a pet but I won't have one
0.501  Dogs are loyal

0.576  Dogs are cool
0.561  I like having a pet but I won't have one
0.501  Dogs are loyal

0.559  A through ball goal is fun
0.383  Attacking midfielder is my favourite position
0.372  Making the run is important for Strikers

0.591  Dogs are loyal
0.579  Golden Retriever's are nice
0.560  Dogs are cool


In [12]:
long_text = "The game of football is a really interesting sport that captures billions of eyes"
small_text = "Everyone likes football"
our_emd = model.encode([long_text,small_text])
print(f"Our Score for cosine sim for these two sentences is :  {sentence_transformers.util.cos_sim(our_emd[0],our_emd[1]).item():.4f}")


Our Score for cosine sim for these two sentences is :  0.5868
